# MGGD Detection Experiment — Colab Runner

Runs `mggd_detection_exp.py` on Colab GPU with Google Drive persistence.

**Workflow:**
1. Run cells 1–4 once per session to set up the environment.
2. Run **Cell 5** to start (or resume) the full experiment.
3. If the session crashes, re-run cells 1–4 then Cell 5 again — `--resume` skips already-finished seeds.
4. Run **Cell 6** at any point to replot from the saved pkl without retraining.
5. Run **Cell 7** to view figures inline.

> **Before first run:** make sure you've pushed your latest local changes to GitHub.
> ```
> git add -A && git commit -m "latest" && git push
> ```

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Clone / pull repo ─────────────────────────────────────────────────
import os
import subprocess

REPO_URL  = 'https://github.com/yonatanshalita-rgb/rao_elliptical_dsm.git'
DRIVE_DIR = '/content/drive/MyDrive/rao_elliptical_dsm'   # change if you prefer a different path

os.makedirs(os.path.dirname(DRIVE_DIR), exist_ok=True)

if not os.path.exists(os.path.join(DRIVE_DIR, '.git')):
    !git clone {REPO_URL} {DRIVE_DIR}
else:
    %cd {DRIVE_DIR}
    !git pull

%cd {DRIVE_DIR}
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# Colab already has torch, numpy, sklearn, matplotlib, scipy.
# PyWavelets may be missing.
!pip install -q PyWavelets

# Create output directories (persisted to Drive)
!mkdir -p checkpoints/det results figures

In [ ]:
# ── Cell 4: Verify GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no GPU detected — experiment will be slower on CPU.')
    print('Go to Runtime > Change runtime type > T4 GPU.')

In [ ]:
# ── Cell 5: Run full experiment ───────────────────────────────────────────────
# --resume skips seeds already recorded in results/mggd_det_results.pkl.
# Re-run this cell after a crash to pick up where you left off.
#
# Expected runtime on T4: ~45–60 min for the full grid.
#
# To run a quick smoke test first (2 seeds, 4 N values):
#   !python mggd_detection_exp.py --quick --n_mc 2

!python mggd_detection_exp.py --resume

In [ ]:
# ── Cell 6: Replot from saved results (no retraining) ────────────────────────
# Use this after the experiment finishes, or to regenerate figures with tweaked
# plot code without re-running training.

!python mggd_detection_exp.py --eval_only

In [ ]:
# ── Cell 7: Display figures inline ────────────────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

for fig in sorted(Path('figures').glob('mggd_det_*.png')):
    print(fig.name)
    display(Image(str(fig)))

In [ ]:
# ── Cell 8: Download figures to local machine (optional) ──────────────────────
# Figures are already saved to Drive. This cell also downloads them directly.
from google.colab import files
from pathlib import Path

for fig in sorted(Path('figures').glob('mggd_det_*.png')):
    files.download(str(fig))

In [ ]:
# ── Cell 9: Print numeric summary from saved pkl ──────────────────────────────
import pickle
import numpy as np

with open('results/mggd_det_results.pkl', 'rb') as f:
    saved = pickle.load(f)

results = saved['results']

METHODS = [
    'Oracle Rao', 'MLE Rao', 'TwoBranch Rao',
    'Linear Rao', 'MGGD Constrained Rao', 'Tyler AMF', 'Oracle Gaussian AMF',
]
N_TRAIN = sorted({k[1] for k in results})
FIXED_SNR = 35.0

print(f'Pd @ Pfa=0.01, SNR={FIXED_SNR} dB')
print(f'{"Method":<25}', end='')
for n in N_TRAIN:
    print(f'  N={n:<6}', end='')
print()
print('-' * (25 + 10 * len(N_TRAIN)))

for method in METHODS:
    print(f'{method:<25}', end='')
    for n in N_TRAIN:
        vals = results.get((method, n, FIXED_SNR), [])
        if not vals:
            print(f'  {"---":<8}', end='')
        else:
            pds = [v[1] for v in vals]
            print(f'  {np.mean(pds):.3f}   ', end='')
    print()